In [1]:
pip install transformers torch pillow

In [2]:
pip install matplotlib scikit-learn

In [4]:
import zipfile

# Путь к zip-архиву и папка для распаковки
zip_path = '/content/plane_data.zip'
extract_folder = '/content/plane_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extract_folder)

print('Архив успешно распакован!')

Архив успешно распакован!


In [15]:
import torch
import os
import glob
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import time

# Проверяем, есть ли GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

CIVILIAN_PROMPTS = [
    "a photo of a large commercial passenger airliner",
    "a Boeing or Airbus passenger jet with airline livery",
    "a civil aviation aircraft at a commercial airport",
    "a small civilian propeller plane like Cessna or Piper",
    "a regional turboprop aircraft",
    "a light civil aviation aircraft",
    "a private business jet",
    "a corporate jet aircraft",
    "a civilian cargo freighter aircraft",
    "a commercial freight transport plane",
    "an aircraft with commercial airline paint scheme"
]

MILITARY_PROMPTS = [
    "a photo of a military fighter",
    "a military bomber aircraft",
    "a combat warplane with weapons",
    "a military transport aircraft",
    "a military cargo plane with camouflage paint",
    "a tactical military transport airplane",
    #"an airborne early warning aircraft like",
    #"a military surveillance and reconnaissance plane",
    "a military aerial refueling tanker aircraft",
    "an aircraft with military camouflage paint scheme",
    "a warplane with military roundels and insignia",
    "a gray painted military aircraft",
]

# Все промпты вместе
ALL_PROMPTS = CIVILIAN_PROMPTS + MILITARY_PROMPTS

# Функция классификации
def classify_image(model, processor, image_path, prompts):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1)[0]

    n_civil = len(CIVILIAN_PROMPTS)

    # max / mean
    civil_score = probs[:n_civil].max().item()
    military_score = probs[n_civil:].max().item()

    # Находим самый уверенный промпт
    best_prompt_idx = probs.argmax().item()
    best_prompt = prompts[best_prompt_idx]
    best_score = probs[best_prompt_idx].item()

    # Определяем категорию
    if civil_score > military_score:
        category = "ГРАЖДАНСКИЙ"
        confidence = civil_score / (civil_score + military_score) * 100
    else:
        category = "ВОЕННЫЙ"
        confidence = military_score / (civil_score + military_score) * 100

    return category, confidence, civil_score, military_score, best_prompt, best_score



def run_classification(model_name, image_paths, true_labels=None):

    # true_labels словарь {путь: "ГРАЖДАНСКИЙ" или "ВОЕННЫЙ"} для подсчёта точности

    print(f"модель: {model_name}")

    # Загружаем модель и процессор
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device)
    model.eval()  # режим инференса

    results = []
    correct, total = 0, 0

    for i, path in enumerate(image_paths, 1):
        filename = os.path.basename(path)

        # Классифицируем
        category, confidence, civil_s, military_s, best_prompt, best_score = classify_image(
            model, processor, path, ALL_PROMPTS
        )

        # Проверяем правильность
        is_correct = ""
        if true_labels and path in true_labels:
            is_correct = "ВЕРНО" if true_labels[path] == category else "!!ОШИБКА!!"
            if is_correct == "!!ОШИБКА!!":
                print(best_prompt)
            if true_labels[path] == category:
                correct += 1
            total += 1

        # Красивый вывод
        print(f"  [{i:2d}/30] {filename:<35s} → {category:<12s} "
              f"(уверенность: {confidence:5.1f}%) {is_correct}")

        results.append({
            "file": filename,
            "prediction": category,
            "confidence": confidence,
            "civil_score": civil_s,
            "military_score": military_s,
        })

    # Итоговая точность
    if total > 0:
        accuracy = correct / total * 100
        print(f"\n Точность: {correct}/{total} = {accuracy:.1f}%")

    return results

# Папка с изображениями
image_dir = "/content/plane_data/plane_data"
valid_extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
image_paths = []
for filename in os.listdir(image_dir):
    ext = os.path.splitext(filename)[1]
    if ext in valid_extensions:
        full_path = os.path.join(image_dir, filename)
        image_paths.append(full_path)
image_paths.sort()

print(f"Найдено изображений: {len(image_paths)}")

TRUE_LABELS = {
    "/content/plane_data/plane_data/Military-Airplane-Show-1-1-scaled.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/01a_aug2018_hansgroenhoff_hgc-1448_live.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/a-10_thunderbolt_ii_supports_operation_epic_fury_9560825.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/3-750x444.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/images.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1151.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1144.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1138.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_113.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1120.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_111.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1100.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1105.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_110.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1103.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1085.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1073.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1023.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1007.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1012.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1004.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1180.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1157.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1156.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/TF_PLANE_1.jpg" : "ГРАЖДАНСКИЙ",
    "/content/plane_data/plane_data/Royal_Air_Force_(RAF)_E-3D_Sentry_AEW1_(airborne_early_warning)_aircraft,_registration_ZH106,_taking_part_in_Exercise_Taurus_Mountain_3,_in_the_skies_over_North_Yorkshire_in_March_2012_-_MOD_45153803_(cropped).jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/49e7a695-0001-0004-0000-000000270346_w520_r1_fpx50_fpy62.86.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/english-electric-lightning-jet-plane-fighter-demobilised-raf-royal-air-force-cold-war-aircraft-at-bruntingthorpe-with-lightning-preservation-group-P58PN6.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/082407a24495e53476c9e7f89d6efd60_0.jpg" : "ВОЕННЫЙ",
    "/content/plane_data/plane_data/028d8d73d48e4368732b3167c4550398_1.jpg" : "ВОЕННЫЙ",
}

if len(image_paths) > 0:

    # Модель маленькая
    results_base = run_classification(
        "openai/clip-vit-base-patch32",
        image_paths,
        TRUE_LABELS if TRUE_LABELS else None
    )

    # Модель большая
    results_large = run_classification(
        "openai/clip-vit-large-patch14",
        image_paths,
        TRUE_LABELS if TRUE_LABELS else None
    )

else:
    print("⏳ Сначала загрузите изображения!")

Используем устройство: cpu
Найдено изображений: 30
модель: openai/clip-vit-base-patch32


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [ 1/30] 01a_aug2018_hansgroenhoff_hgc-1448_live.jpg → ВОЕННЫЙ      (уверенность:  95.3%) ВЕРНО
  [ 2/30] 028d8d73d48e4368732b3167c4550398_1.jpg → ВОЕННЫЙ      (уверенность:  87.3%) ВЕРНО
  [ 3/30] 082407a24495e53476c9e7f89d6efd60_0.jpg → ВОЕННЫЙ      (уверенность:  97.2%) ВЕРНО
  [ 4/30] 3-750x444.jpg                       → ВОЕННЫЙ      (уверенность:  95.3%) ВЕРНО
  [ 5/30] 49e7a695-0001-0004-0000-000000270346_w520_r1_fpx50_fpy62.86.jpg → ВОЕННЫЙ      (уверенность:  96.4%) ВЕРНО
  [ 6/30] Military-Airplane-Show-1-1-scaled.jpg → ВОЕННЫЙ      (уверенность:  97.6%) ВЕРНО
  [ 7/30] Royal_Air_Force_(RAF)_E-3D_Sentry_AEW1_(airborne_early_warning)_aircraft,_registration_ZH106,_taking_part_in_Exercise_Taurus_Mountain_3,_in_the_skies_over_North_Yorkshire_in_March_2012_-_MOD_45153803_(cropped).jpg → ВОЕННЫЙ      (уверенность:  97.5%) ВЕРНО
  [ 8/30] TF_PLANE_1.jpg                      → ГРАЖДАНСКИЙ  (уверенность:  95.6%) ВЕРНО
  [ 9/30] TF_PLANE_1004.jpg                   → ГРАЖДАНСКИЙ  (увер

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

  [ 1/30] 01a_aug2018_hansgroenhoff_hgc-1448_live.jpg → ВОЕННЫЙ      (уверенность:  99.5%) ВЕРНО
  [ 2/30] 028d8d73d48e4368732b3167c4550398_1.jpg → ВОЕННЫЙ      (уверенность:  94.6%) ВЕРНО
  [ 3/30] 082407a24495e53476c9e7f89d6efd60_0.jpg → ВОЕННЫЙ      (уверенность:  90.7%) ВЕРНО
  [ 4/30] 3-750x444.jpg                       → ВОЕННЫЙ      (уверенность:  99.0%) ВЕРНО
  [ 5/30] 49e7a695-0001-0004-0000-000000270346_w520_r1_fpx50_fpy62.86.jpg → ВОЕННЫЙ      (уверенность:  93.9%) ВЕРНО
  [ 6/30] Military-Airplane-Show-1-1-scaled.jpg → ВОЕННЫЙ      (уверенность:  99.6%) ВЕРНО
  [ 7/30] Royal_Air_Force_(RAF)_E-3D_Sentry_AEW1_(airborne_early_warning)_aircraft,_registration_ZH106,_taking_part_in_Exercise_Taurus_Mountain_3,_in_the_skies_over_North_Yorkshire_in_March_2012_-_MOD_45153803_(cropped).jpg → ВОЕННЫЙ      (уверенность:  99.5%) ВЕРНО
  [ 8/30] TF_PLANE_1.jpg                      → ГРАЖДАНСКИЙ  (уверенность:  91.0%) ВЕРНО
  [ 9/30] TF_PLANE_1004.jpg                   → ГРАЖДАНСКИЙ  (увер